In [1]:
from QuadraticRoughHeston import QuadraticRoughHeston

from tools.grid import *
from tools.qrh_converge_test import *
from tools.qrh_params import *
from fx_init_const import *

In [2]:
from fwd_var_curve import xi_curve_smooth, var_swap_robust

In [3]:
date = '01-12-2023'

In [4]:
img_date_path = f"{date}/img/"
data_date_path = f"{date}/data/"
img_expiries_path = f"{img_date_path}{TAU_STR[0]}-to-{TAU_STR[-1]}/"
data_expiries_path = f"{data_date_path}{TAU_STR[0]}-to-{TAU_STR[-1]}/"
ig_path = f"{img_expiries_path}initial_guess/"
mkt_path = f"{img_expiries_path}market/"
opt_path = f"{img_expiries_path}optimised/"

setup_dict = setup(
    volatility_grid=VOL_QUOTES,
    expiries=TAU,
    foreign_rate=USD_OIS,
    domestic_rate=EUR_OIS,
    forward_prices=FWD,
    spot=spot,
    data_expiries_path=data_expiries_path,
)

fx_df = setup_dict["fx_df"]
strike_grid = setup_dict["strike_grid"]
PRICE_FROM_VOL_QUOTES = setup_dict["price_grid"]
log_fwd_moneyness_grid = setup_dict["log_fwd_moneyness_grid"]
inverse_vol_grid = setup_dict["inverse_vol_grid"]
base_rates_arr = setup_dict["base_rates_arr"]
term_rates_arr = setup_dict["term_rates_arr"]
cp_flags_grid = setup_dict["cp_flags_grid"]

In [5]:
fx_var_swap = var_swap_robust(fx_df, TAU)
fx_expiries_arr = fx_var_swap["expiries"]
w_in = fx_var_swap["vs_mid"] * fx_expiries_arr
xi_smooth = xi_curve_smooth(fx_expiries_arr, w_in, eps=0.03)["xi_curve"]

Processing slice: 0 (TAU=0.0027397260273972603)
Processing slice: 1 (TAU=0.019230769230769232)
Processing slice: 2 (TAU=0.038461538461538464)
Processing slice: 3 (TAU=0.057692307692307696)
Processing slice: 4 (TAU=0.08333333333333333)
Processing slice: 5 (TAU=0.16666666666666666)
Processing slice: 6 (TAU=0.25)
Processing slice: 7 (TAU=0.3333333333333333)
Processing slice: 8 (TAU=0.4166666666666667)
Processing slice: 9 (TAU=0.5)
Processing slice: 10 (TAU=0.75)
Processing slice: 11 (TAU=1.0)
Processing slice: 12 (TAU=1.5)
Processing slice: 13 (TAU=2.0)
Processing slice: 14 (TAU=3.0)
Processing slice: 15 (TAU=4.0)
Processing slice: 16 (TAU=5.0)


In [6]:
COMBINED_OIS_DICT = {expiry: rate for (expiry, rate) in zip(fx_expiries_arr, COMBINED_OIS)}

In [7]:
PATHS = 10000
STEPS = 2500

In [8]:
qrh_params = {
    'c': 5.815452e-04,
    'nu': 0.249717,
    'lam': 1.105411,
    'al': 0.511054,
    'a': 0.693534,
    'b': 2.727018
}
print(qrh_params)

qrh = QuadraticRoughHeston(**qrh_params, xi0=xi_smooth)
res = qrh.simulate_filtered_random(PATHS, STEPS, fx_var_swap["expiries"], interest_rates=COMBINED_OIS_DICT)

{'c': 0.0005815452, 'nu': 0.249717, 'lam': 1.105411, 'al': 0.511054, 'a': 0.693534, 'b': 2.727018}


/Users/anthonynkyi/Documents/cs/Finalyear/calibrationFX/QuadraticRoughHeston.py:108: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  return integrate.quad(lambda x: self.resolvent_kernel(x), 0.0, x)[0]


In [9]:
import numpy as np
mc_path_matrix_new = np.array([res[expiry]["X"] for expiry in fx_expiries_arr])
mc_var_matrix_new = np.array([res[expiry]["V"] for expiry in fx_expiries_arr])

In [10]:
model_price_grid = get_mc_prices_grid_log_fwd(
    mc_path_matrix_new, spot, log_fwd_moneyness_grid, fx_expiries_arr, base_rates_arr, term_rates_arr, cp_flags_grid, FWD
)
iv_grid_jaeckel = get_iv_from_prices_grid_jaeckel(
    model_price_grid, spot, strike_grid, fx_expiries_arr, base_rates_arr, term_rates_arr, cp_flags_grid,
)
iv_grid_gatheral = get_iv_from_paths_grid_gatheral(
    mc_path_matrix_new, spot, log_fwd_moneyness_grid, fx_expiries_arr, base_rates_arr, term_rates_arr, cp_flags_grid
)

np.savetxt(f"{data_expiries_path}model_price_grid.csv", model_price_grid, delimiter=",")
np.savetxt(f"{data_expiries_path}iv_grid_jaeckel.csv", iv_grid_jaeckel, delimiter=",")
np.savetxt(f"{data_expiries_path}iv_grid_gatheral.csv", iv_grid_gatheral, delimiter=",")